# Faz 8 - CapERA test-only real embeddings

Quality scope is exactly 1391 test videos and 6955 caption queries. The source has no caption provenance, therefore caption_source is unknown. No synthetic fallback is used.

In [ ]:
%pip install -q 'transformers>=4.57.3' 'accelerate>=1.12.0' 'qwen-vl-utils>=0.0.14' huggingface-hub pyarrow pandas numpy
!test -d /content/Multimodal-Video-Intelligence || git clone --depth 1 https://github.com/ColdVI/Multimodal-Video-Intelligence.git /content/Multimodal-Video-Intelligence
!test -d /content/Qwen3-VL-Embedding || git clone --depth 1 https://github.com/QwenLM/Qwen3-VL-Embedding.git /content/Qwen3-VL-Embedding
import sys
sys.path[:0] = ['/content/Multimodal-Video-Intelligence', '/content/Qwen3-VL-Embedding']

In [ ]:
import subprocess, torch
if not torch.cuda.is_available():
    raise SystemExit('GPU missing: choose a GPU runtime and restart.')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
GPU_NAME = torch.cuda.get_device_name(0)
major, _ = torch.cuda.get_device_capability(0)
TORCH_DTYPE = torch.float16 if major < 8 else torch.bfloat16
ATTN_IMPL = 'sdpa'
print({'gpu': GPU_NAME, 'dtype': str(TORCH_DTYPE), 'attention': ATTN_IMPL})

In [ ]:
from huggingface_hub import model_info, snapshot_download
from src.models.qwen3_vl_embedding import Qwen3VLEmbedder
MODEL_ID = 'Qwen/Qwen3-VL-Embedding-2B'
MODEL_REVISION = model_info(MODEL_ID).sha
MODEL_PATH = snapshot_download(MODEL_ID, revision=MODEL_REVISION)
model = Qwen3VLEmbedder(model_name_or_path=MODEL_PATH, fps=1.0, max_frames=8, max_length=16384, torch_dtype=TORCH_DTYPE, attn_implementation=ATTN_IMPL)
print({'model': MODEL_ID, 'revision': MODEL_REVISION})

In [ ]:
import json, os, zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Multimodal-Video-Intelligence')
INPUT = DRIVE_ROOT / 'datasets' / 'capera'
OUT = DRIVE_ROOT / 'artifacts' / 'embeddings'
OUT.mkdir(parents=True, exist_ok=True)
ERA_ZIP = INPUT / 'ERA_Dataset.zip'
TRAIN_JSON = INPUT / 'CapERA_DATASET_train.json'
TEST_JSON = INPUT / 'CapERA_DATASET_test.json'
for path in (ERA_ZIP, TRAIN_JSON, TEST_JSON):
    if not path.exists(): raise FileNotFoundError(path)
extract_root = Path('/content/capera_extract')
if not extract_root.exists():
    extract_root.mkdir(parents=True)
    with zipfile.ZipFile(ERA_ZIP) as archive:
        root = extract_root.resolve()
        for member in archive.infolist():
            target = (extract_root / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(extract_root)
VIDEO_ROOT = next((p for p in (extract_root/'ERA_Dataset'/'Videos', extract_root/'Videos') if p.exists()), None)
if VIDEO_ROOT is None: raise FileNotFoundError('Videos/Test and Videos/Tra not found')
print({'video_root': str(VIDEO_ROOT), 'out': str(OUT)})

In [ ]:
import copy
from common import load_config
from dataset_adapters.capera import CapERAAdapter
cfg = copy.deepcopy(load_config('/content/Multimodal-Video-Intelligence/config.yaml'))
cfg['datasets']['capera'].update({'train_split': str(TRAIN_JSON), 'test_split': str(TEST_JSON), 'videos_dir': str(VIDEO_ROOT)})
adapter = CapERAAdapter(cfg=cfg)
sequence_ids = adapter.list_sequences(split='test')
assert len(sequence_ids) == 1391, len(sequence_ids)
items, queries = [], []
for video_id in sequence_ids:
    captions = adapter.captions(video_id)
    assert len(captions) == 5, (video_id, len(captions))
    video_path = adapter.load_video(video_id)
    if not video_path.exists(): raise FileNotFoundError(video_path)
    segment_id = f'capera:{video_id}:0.000:5.000'
    items.append({'segment_id': segment_id, 'video_id': video_id, 'video_path': str(video_path)})
    for caption_index, text in enumerate(captions):
        queries.append({'query_id': f'capera:{video_id}:caption:{caption_index}', 'query_text': text, 'relevant_segment_id': segment_id, 'relevant_video_id': video_id, 'caption_index': caption_index, 'caption_source': 'unknown'})
assert len(items) == 1391 and len(queries) == 6955
print({'items': len(items), 'queries': len(queries), 'source': 'unknown'})

In [ ]:
import time, numpy as np, pandas as pd
CHECKPOINT_EVERY = 200
def normalize(value):
    value = np.asarray(value, dtype=np.float32)
    value /= np.linalg.norm(value)
    if value.shape != (2048,) or not np.isfinite(value).all(): raise ValueError(value.shape)
    return value
def resume(name, count):
    partial = OUT / f'{name}.partial.npy'
    done_path = OUT / f'{name}.done.json'
    if partial.exists():
        matrix = np.lib.format.open_memmap(partial, mode='r+')
        assert matrix.shape == (count, 2048)
    else:
        matrix = np.lib.format.open_memmap(partial, mode='w+', dtype=np.float32, shape=(count, 2048)); matrix[:] = np.nan
    done = set(json.loads(done_path.read_text()) if done_path.exists() else [])
    return partial, done_path, matrix, done
def flush(matrix, done_path, done):
    matrix.flush(); temporary = done_path.with_suffix('.tmp')
    temporary.write_text(json.dumps(sorted(done)), encoding='utf-8')
    os.replace(temporary, done_path)

In [ ]:
item_partial, item_done_path, item_vectors, item_done = resume('capera_2048', 1391)
item_started = time.perf_counter(); since = 0
for position, row in enumerate(items):
    if row['segment_id'] in item_done: continue
    output = model.process([{'video': row['video_path'], 'fps': 1.0, 'max_frames': 8}])
    item_vectors[position] = normalize(output[0].detach().cpu().float().numpy())
    item_done.add(row['segment_id']); since += 1
    if since >= CHECKPOINT_EVERY:
        flush(item_vectors, item_done_path, item_done); since = 0
        print({'item_checkpoint': len(item_done)})
flush(item_vectors, item_done_path, item_done)
assert len(item_done) == 1391
ITEM_ELAPSED_S = time.perf_counter() - item_started

In [ ]:
query_partial, query_done_path, query_vectors, query_done = resume('capera_queries_2048', 6955)
query_started = time.perf_counter()
pending = [(i, row) for i, row in enumerate(queries) if row['query_id'] not in query_done]
last_checkpoint = len(query_done)
for start in range(0, len(pending), 16):
    batch = pending[start:start+16]
    output = model.process([{'text': row['query_text']} for _, row in batch])
    for (position, row), vector in zip(batch, output):
        query_vectors[position] = normalize(vector.detach().cpu().float().numpy())
        query_done.add(row['query_id'])
    if len(query_done) - last_checkpoint >= CHECKPOINT_EVERY:
        flush(query_vectors, query_done_path, query_done); last_checkpoint = len(query_done)
        print({'query_checkpoint': len(query_done)})
flush(query_vectors, query_done_path, query_done)
assert len(query_done) == 6955
QUERY_ELAPSED_S = time.perf_counter() - query_started

In [ ]:
semantic = json.loads(Path('/content/Multimodal-Video-Intelligence/tests/fixtures/queries_semantic.json').read_text(encoding='utf-8'))
demo_texts = []
for row in semantic:
    if 'generator' in row:
        demo_texts.append(row['generator']['character'] * row['generator']['length'])
    else:
        demo_texts.extend(value for value in (row.get('tr'), row.get('en')) if value)
demo_texts = list(dict.fromkeys(demo_texts))
outputs = model.process([{'text': text} for text in demo_texts])
fixed = {text: normalize(vector.detach().cpu().float().numpy()).tolist() for text, vector in zip(demo_texts, outputs)}
(OUT/'query_embeddings.json').write_text(json.dumps({'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'queries': fixed}, ensure_ascii=False), encoding='utf-8')
print({'fixed_demo_queries': len(fixed), 'all_6955_in_json': False})

In [ ]:
import hashlib, shutil
item_final, query_final = OUT/'capera_2048.npy', OUT/'capera_queries_2048.npy'
shutil.copy2(item_partial, item_final); shutil.copy2(query_partial, query_final)
pd.DataFrame(items)[['segment_id','video_id']].to_parquet(OUT/'capera_ids.parquet', index=False)
pd.DataFrame(queries).to_parquet(OUT/'capera_query_ids.parquet', index=False)
iv, qv = np.load(item_final, mmap_mode='r'), np.load(query_final, mmap_mode='r')
assert iv.shape == (1391,2048) and qv.shape == (6955,2048)
assert iv.dtype == qv.dtype == np.float32 and np.isfinite(iv).all() and np.isfinite(qv).all()
assert np.allclose(np.linalg.norm(iv,axis=1),1,atol=1e-5)
assert np.allclose(np.linalg.norm(qv,axis=1),1,atol=1e-5)
manifest = {'dataset_id':'capera','split':'test','item_count':1391,'query_count':6955,'captions_per_item':5,'caption_source':'unknown','dimension':2048,'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'dtype':str(TORCH_DTYPE),'attention_implementation':ATTN_IMPL,'gpu_name':GPU_NAME,'frames_per_item':8,'embedding_mode':'real','item_elapsed_s':ITEM_ELAPSED_S,'query_elapsed_s':QUERY_ELAPSED_S,'test_caption_sha256':hashlib.sha256(TEST_JSON.read_bytes()).hexdigest()}
(OUT/'embedding_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print(manifest)

In [ ]:
from google.colab import files
names = ['capera_2048.npy','capera_ids.parquet','capera_queries_2048.npy','capera_query_ids.parquet','query_embeddings.json','embedding_manifest.json']
zip_path = Path('/content/capera_embeddings_faz8.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for name in names: archive.write(OUT/name,arcname=name)
shutil.copy2(zip_path, DRIVE_ROOT/zip_path.name)
print({'zip':str(zip_path),'files':names}); files.download(str(zip_path))